In [1]:
import kagglehub
path = kagglehub.dataset_download("jessicali9530/animal-crossing-new-horizons-nookplaza-dataset")

Using Colab cache for faster access to the 'animal-crossing-new-horizons-nookplaza-dataset' dataset.


In [2]:
import os

# List the contents of the downloaded directory
print(f"Contents of the dataset directory: {path}")
for dirname, _, filenames in os.walk(path):
    for filename in filenames:
        print(os.path.join(dirname, filename))

Contents of the dataset directory: /kaggle/input/animal-crossing-new-horizons-nookplaza-dataset
/kaggle/input/animal-crossing-new-horizons-nookplaza-dataset/construction.csv
/kaggle/input/animal-crossing-new-horizons-nookplaza-dataset/bottoms.csv
/kaggle/input/animal-crossing-new-horizons-nookplaza-dataset/accessories.csv
/kaggle/input/animal-crossing-new-horizons-nookplaza-dataset/insects.csv
/kaggle/input/animal-crossing-new-horizons-nookplaza-dataset/floors.csv
/kaggle/input/animal-crossing-new-horizons-nookplaza-dataset/shoes.csv
/kaggle/input/animal-crossing-new-horizons-nookplaza-dataset/wall-mounted.csv
/kaggle/input/animal-crossing-new-horizons-nookplaza-dataset/achievements.csv
/kaggle/input/animal-crossing-new-horizons-nookplaza-dataset/umbrellas.csv
/kaggle/input/animal-crossing-new-horizons-nookplaza-dataset/rugs.csv
/kaggle/input/animal-crossing-new-horizons-nookplaza-dataset/housewares.csv
/kaggle/input/animal-crossing-new-horizons-nookplaza-dataset/art.csv
/kaggle/input/

In [3]:
import pandas as pd

# Assuming 'path' variable still holds the dataset directory
housewares_df = pd.read_csv(f"{path}/housewares.csv")

print("First 5 rows of housewares_df:")
print(housewares_df.head())

print("\nInformation about housewares_df:")
housewares_df.info()

First 5 rows of housewares_df:
              Name Variation Body Title Pattern Pattern Title  DIY  \
0  acoustic guitar   Natural       Body     NaN           NaN  Yes   
1  acoustic guitar    Cherry       Body     NaN           NaN  Yes   
2  acoustic guitar     Brown       Body     NaN           NaN  Yes   
3  acoustic guitar      Blue       Body     NaN           NaN  Yes   
4  acoustic guitar     White       Body     NaN           NaN  Yes   

  Body Customize Pattern Customize  Kit Cost  Buy  ...  Interact  \
0            Yes                No       5.0  NFS  ...       Yes   
1            Yes                No       5.0  NFS  ...       Yes   
2            Yes                No       5.0  NFS  ...       Yes   
3            Yes                No       5.0  NFS  ...       Yes   
4            Yes                No       5.0  NFS  ...       Yes   

                  Tag Outdoor         Speaker Type  Lighting Type  \
0  Musical Instrument      No  Does not play music    No lighting   
1

### Model 1: Predicting 'Sell' Price (Regression)

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import numpy as np

# Make a copy to avoid modifying the original DataFrame
df_sell = housewares_df.copy()

# Drop columns that are not relevant for prediction or have too many missing values
# or are redundant identifiers.
drop_columns = [
    'Body Title', 'Pattern', 'Pattern Title', 'Miles Price', # Too many NaNs or not directly relevant for sell price
    'Source Notes', 'HHA Concept 2', 'HHA Series', 'HHA Set', # Too many NaNs
    'Filename', 'Variant ID', 'Internal ID', 'Unique Entry ID', # Identifiers
    'Buy' # 'Buy' column contains 'NFS' (Not For Sale) which complicates direct numerical conversion
]

df_sell = df_sell.drop(columns=drop_columns, errors='ignore')

# Handle the 'Buy' column values for 'NFS' items
# For simplicity, we can assume 'NFS' items have a 'Sell' price of 0 if we decide to keep them in the dataset.
# Or, we can filter them out if we only want to predict for sellable items.
# For now, let's filter out 'NFS' items for price prediction
df_sell = df_sell[df_sell['Catalog'] != 'Not for sale'].copy()

# Convert 'Sell' column to numeric, handling potential non-numeric values if any remain
df_sell['Sell'] = pd.to_numeric(df_sell['Sell'], errors='coerce')

# Drop rows where 'Sell' price is NaN (if any occurred due to 'coerce')
df_sell.dropna(subset=['Sell'], inplace=True)

print(f"Shape of data after initial cleaning: {df_sell.shape}")
print("Columns and their non-null counts after cleaning:")
df_sell.info()

Shape of data after initial cleaning: (1870, 19)
Columns and their non-null counts after cleaning:
<class 'pandas.core.frame.DataFrame'>
Index: 1870 entries, 7 to 3269
Data columns (total 19 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Name               1870 non-null   object 
 1   Variation          1765 non-null   object 
 2   DIY                1870 non-null   object 
 3   Body Customize     1870 non-null   object 
 4   Pattern Customize  1870 non-null   object 
 5   Kit Cost           1003 non-null   float64
 6   Sell               1870 non-null   int64  
 7   Color 1            1870 non-null   object 
 8   Color 2            1870 non-null   object 
 9   Size               1870 non-null   object 
 10  Source             1870 non-null   object 
 11  Version            1870 non-null   object 
 12  HHA Concept 1      1870 non-null   object 
 13  Interact           1870 non-null   object 
 14  Tag                1870 no

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer # Added this import
import numpy as np

# Define target and features
X = df_sell.drop('Sell', axis=1)
y = df_sell['Sell']

# Identify categorical and numerical features
categorical_features = X.select_dtypes(include=['object']).columns
numerical_features = X.select_dtypes(include=['float64', 'int64']).columns

# Preprocessing pipelines for numerical and categorical features
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Create a column transformer to apply different transformations to different columns
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Create the full preprocessing pipeline
preprocessing_pipeline = Pipeline(steps=[('preprocessor', preprocessor)])

# Apply preprocessing to the features
X_processed = preprocessing_pipeline.fit_transform(X)

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.2, random_state=42)

print(f"Shape of processed features: {X_processed.shape}")
print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")

Shape of processed features: (1870, 522)
Shape of X_train: (1496, 522)
Shape of X_test: (374, 522)


In [7]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Define the deep learning model (a simple feedforward neural network)
model_sell = keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1) # Output layer for regression (single numerical output)
])

# Compile the model
model_sell.compile(optimizer='adam', loss='mse', metrics=['mae'])

# Train the model
history_sell = model_sell.fit(
    X_train,
    y_train,
    epochs=50, # You can adjust the number of epochs
    batch_size=32,
    validation_split=0.2, # Use a portion of training data for validation
    verbose=1
)

print("\nModel training complete.")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - loss: 57623748.0000 - mae: 2858.8843 - val_loss: 56228900.0000 - val_mae: 2699.5320
Epoch 2/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 57449352.0000 - mae: 2828.7720 - val_loss: 55847692.0000 - val_mae: 2631.3440
Epoch 3/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 56686604.0000 - mae: 2695.6299 - val_loss: 54612852.0000 - val_mae: 2425.1880
Epoch 4/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 54762984.0000 - mae: 2479.0088 - val_loss: 52392732.0000 - val_mae: 2262.6343
Epoch 5/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 52201544.0000 - mae: 2423.8213 - val_loss: 49816736.0000 - val_mae: 2395.4021
Epoch 6/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 49779308.0000 - mae: 2683.5862 - val_loss: 47814452.0000 - val_mae: 2746.9355
Epoch 7/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 48029864.0000 - mae: 2969.4585 - val_loss: 46539652.0000 - val_mae: 3042.6172
Epoch 8/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms

In [32]:
import joblib
import os
import pandas as pd

# Create a directory to save model artifacts if it doesn't exist
model_dir = 'model_artifacts'
os.makedirs(model_dir, exist_ok=True)

# Save the preprocessing pipeline
joblib.dump(preprocessing_pipeline, os.path.join(model_dir, 'preprocessing_pipeline_sell.pkl'))

# Save the trained Keras model
model_sell.save(os.path.join(model_dir, 'model_sell_price.keras'))

# --- Save metadata for Streamlit app ---
# original_features was defined relative to df_sell in cell e2904ac4
original_features_for_streamlit = [col for col in df_sell.columns if col != 'Sell']
joblib.dump(original_features_for_streamlit, os.path.join(model_dir, 'original_features_sell.pkl'))

# Collect unique values for categorical features from df_sell for Streamlit
unique_categorical_values_sell = {}
# Use the categorical_features list from the preprocessing step to ensure consistency
for feature in categorical_features:
    if feature in df_sell.columns:
        unique_categorical_values_sell[feature] = df_sell[feature].dropna().unique().tolist()
joblib.dump(unique_categorical_values_sell, os.path.join(model_dir, 'unique_cat_values_sell.pkl'))

# Also save the mean of 'Kit Cost' for default value in Streamlit
kit_cost_mean = df_sell['Kit Cost'].mean()
joblib.dump(kit_cost_mean, os.path.join(model_dir, 'kit_cost_mean.pkl'))

print("Preprocessing pipeline, model, and Streamlit metadata saved successfully.")

Preprocessing pipeline, model, and Streamlit metadata saved successfully.


### Streamlit Application for 'Sell' Price Prediction

Below is the code for a Streamlit application. To run this app, save the code to a file (e.g., `streamlit_app.py`) and then execute `streamlit run streamlit_app.py` in your terminal. This application will load the saved preprocessing pipeline and the trained deep learning model to predict the 'Sell' price of a houseware item based on user input.

In [42]:
%%writefile streamlit_app.py

import streamlit as st
import pandas as pd
import joblib
import numpy as np
import tensorflow as tf

# Load artifacts for Sell Price Prediction Model
@st.cache_resource
def load_sell_artifacts():
    pipeline = joblib.load('model_artifacts/preprocessing_pipeline_sell.pkl')
    model = tf.keras.models.load_model('model_artifacts/model_sell_price.keras')
    original_features = joblib.load('model_artifacts/original_features_sell.pkl')
    unique_cat_values = joblib.load('model_artifacts/unique_cat_values_sell.pkl')
    kit_cost_mean = joblib.load('model_artifacts/kit_cost_mean.pkl')
    return pipeline, model, original_features, unique_cat_values, kit_cost_mean

# Load artifacts for DIY Classification Model
@st.cache_resource
def load_diy_artifacts():
    pipeline = joblib.load('model_artifacts/preprocessing_pipeline_diy.pkl')
    model = tf.keras.models.load_model('model_artifacts/model_diy.keras')
    original_features = joblib.load('model_artifacts/original_features_diy.pkl')
    unique_cat_values = joblib.load('model_artifacts/unique_cat_values_diy.pkl')
    kit_cost_mean = joblib.load('model_artifacts/kit_cost_mean_diy.pkl')
    return pipeline, model, original_features, unique_cat_values, kit_cost_mean

# Load all models and preprocessing pipelines
preprocessing_pipeline_sell, model_sell, original_features_sell, unique_categorical_values_sell, kit_cost_mean_sell = load_sell_artifacts()
preprocessing_pipeline_diy, model_diy, original_features_diy, unique_categorical_values_diy, kit_cost_mean_diy = load_diy_artifacts()

# --- Page Configuration and Title ---
st.set_page_config(page_title="ACNH Houseware Predictor", page_icon="🏡", layout="centered")

st.title('🏡 Animal Crossing Houseware Predictor')
st.markdown("**_Predict item 'Sell' prices or classify if they are 'DIY' items!_**")

# --- Tabbed Interface ---
tab1, tab2 = st.tabs(["💰 Sell Price Prediction", "🛠️ DIY Classification"])

with tab1:
    st.header('💰 Predict "Sell" Price')
    st.markdown("Enter the details of a houseware item to predict its 'Sell' price.")

    # Streamlit input widgets for Sell Price Prediction features
    input_data_sell = {}

    # Numerical features for Sell model
    num_features_to_input_sell = ['Kit Cost']
    for feature in num_features_to_input_sell:
        default_value = float(kit_cost_mean_sell)
        input_data_sell[feature] = st.number_input(f'Sell - Enter {feature}', value=default_value, format="%.2f")

    # Categorical features for Sell model
    cat_features_to_input_sell = [f for f in original_features_sell if f not in num_features_to_input_sell]

    for feature in cat_features_to_input_sell:
        options = unique_categorical_values_sell.get(feature, [])
        if options:
            default_index = 0
            if len(options) > 0 and options[0] in options:
                default_index = options.index(options[0])
            input_data_sell[feature] = st.selectbox(f'Sell - Select {feature}', options, index=default_index, key=f'sell_{feature}')
        else:
            st.write(f"Warning: No options found for categorical feature: {feature}. Using text input.")
            input_data_sell[feature] = st.text_input(f'Sell - Enter {feature}', key=f'sell_{feature}')

    # --- Prediction Button for Sell Price ---
    if st.button('🔮 Predict Sell Price', key='predict_sell_button'):
        st.snow()
        input_df_sell = pd.DataFrame([input_data_sell], columns=original_features_sell)

        try:
            processed_input_sell = preprocessing_pipeline_sell.transform(input_df_sell)
            prediction_sell = model_sell.predict(processed_input_sell)[0][0]
            st.success(f'The predicted Sell Price is: **${prediction_sell:,.2f}** 🔔')
        except Exception as e:
            st.error(f"An error occurred during Sell Price prediction: {e}")
            st.write("Please ensure all input fields are correctly filled and match expected types for Sell Price Prediction.")

with tab2:
    st.header('🛠️ Classify "DIY" Item')
    st.markdown("Enter the details of a houseware item to classify if it is a 'DIY' item.")

    # Streamlit input widgets for DIY Classification features
    input_data_diy = {}

    # Numerical features for DIY model
    num_features_to_input_diy = ['Kit Cost']
    for feature in num_features_to_input_diy:
        default_value = float(kit_cost_mean_diy)
        input_data_diy[feature] = st.number_input(f'DIY - Enter {feature}', value=default_value, format="%.2f", key=f'diy_{feature}')

    # Categorical features for DIY model
    cat_features_to_input_diy = [f for f in original_features_diy if f not in num_features_to_input_diy]

    for feature in cat_features_to_input_diy:
        options = unique_categorical_values_diy.get(feature, [])
        if options:
            default_index = 0
            if len(options) > 0 and options[0] in options:
                default_index = options.index(options[0])
            input_data_diy[feature] = st.selectbox(f'DIY - Select {feature}', options, index=default_index, key=f'diy_{feature}')
        else:
            st.write(f"Warning: No options found for categorical feature: {feature}. Using text input.")
            input_data_diy[feature] = st.text_input(f'DIY - Enter {feature}', key=f'diy_{feature}')

    # --- Prediction Button for DIY Classification ---
    if st.button('🔮 Classify DIY Item', key='predict_diy_button'):
        st.snow()
        input_df_diy = pd.DataFrame([input_data_diy], columns=original_features_diy)

        try:
            processed_input_diy = preprocessing_pipeline_diy.transform(input_df_diy)
            prediction_diy_proba = model_diy.predict(processed_input_diy)[0][0]
            prediction_diy_class = "Yes, it's a DIY item! 🔨" if prediction_diy_proba >= 0.5 else "No, it's not a DIY item. 🛍️"
            st.success(f'DIY Classification: **{prediction_diy_class}** (Probability: {prediction_diy_proba:.2f})')
        except Exception as e:
            st.error(f"An error occurred during DIY Classification: {e}")
            st.write("Please ensure all input fields are correctly filled and match expected types for DIY Classification.")

st.markdown("--- ")
st.info("Note: These models predict based on the 'housewares.csv' dataset. Predictions are estimates.")


Overwriting streamlit_app.py


### Model 2: Classifying 'DIY' Items (Binary Classification)

In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Make a copy for DIY classification task
df_diy = housewares_df.copy()

# Define the target variable for DIY classification
y_diy = df_diy['DIY'].apply(lambda x: 1 if x == 'Yes' else 0) # Convert 'Yes'/'No' to 1/0

# Drop target and other irrelevant columns for this task
drop_columns_diy = [
    'DIY', 'Sell', 'Buy', # Target and other task-specific columns
    'Body Title', 'Pattern', 'Pattern Title', 'Miles Price', # Too many NaNs or not directly relevant
    'Source Notes', 'HHA Concept 2', 'HHA Series', 'HHA Set', # Too many NaNs
    'Filename', 'Variant ID', 'Internal ID', 'Unique Entry ID', # Identifiers
    'Catalog' # Catalog might be highly correlated with DIY or not useful
]

df_diy = df_diy.drop(columns=drop_columns_diy, errors='ignore')

# Separate features (X_diy)
X_diy = df_diy

# Identify categorical and numerical features for this specific task
categorical_features_diy = X_diy.select_dtypes(include=['object']).columns
numerical_features_diy = X_diy.select_dtypes(include=['float64', 'int64']).columns

# Preprocessing pipelines (same as before, but specific to this task)
numerical_transformer_diy = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

categorical_transformer_diy = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor_diy = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer_diy, numerical_features_diy),
        ('cat', categorical_transformer_diy, categorical_features_diy)
    ])

preprocessing_pipeline_diy = Pipeline(steps=[('preprocessor', preprocessor_diy)])

# Apply preprocessing to the features
X_diy_processed = preprocessing_pipeline_diy.fit_transform(X_diy)

# Split data into training and testing sets
X_train_diy, X_test_diy, y_train_diy, y_test_diy = train_test_split(X_diy_processed, y_diy, test_size=0.2, random_state=42, stratify=y_diy)

print(f"Shape of processed features for DIY model: {X_diy_processed.shape}")
print(f"Shape of X_train_diy: {X_train_diy.shape}")
print(f"Shape of X_test_diy: {X_test_diy.shape}")
print(f"Distribution of 'DIY' in y_train_diy:\n{y_train_diy.value_counts()}")
print(f"Distribution of 'DIY' in y_test_diy:\n{y_test_diy.value_counts()}")

Shape of processed features for DIY model: (3275, 941)
Shape of X_train_diy: (2620, 941)
Shape of X_test_diy: (655, 941)
Distribution of 'DIY' in y_train_diy:
DIY
0    1642
1     978
Name: count, dtype: int64
Distribution of 'DIY' in y_test_diy:
DIY
0    411
1    244
Name: count, dtype: int64


In [39]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import joblib
import os

# Define the deep learning model for DIY classification
model_diy = keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(X_train_diy.shape[1],)),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid') # Output layer for binary classification
])

# Compile the model
model_diy.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
history_diy = model_diy.fit(
    X_train_diy,
    y_train_diy,
    epochs=50, # You can adjust the number of epochs
    batch_size=32,
    validation_split=0.2, # Use a portion of training data for validation
    verbose=1
)

print("\nDIY model training complete.")

# Create a directory to save model artifacts if it doesn't exist
model_dir = 'model_artifacts'
os.makedirs(model_dir, exist_ok=True)

# Save the preprocessing pipeline for DIY model
joblib.dump(preprocessing_pipeline_diy, os.path.join(model_dir, 'preprocessing_pipeline_diy.pkl'))

# Save the trained Keras DIY model
model_diy.save(os.path.join(model_dir, 'model_diy.keras'))

# --- Save metadata for Streamlit app for DIY model ---
# original_features for DIY was defined relative to df_diy after dropping target and other columns
# X_diy contains the features used for training before preprocessing
original_features_for_streamlit_diy = X_diy.columns.tolist()
joblib.dump(original_features_for_streamlit_diy, os.path.join(model_dir, 'original_features_diy.pkl'))

# Collect unique values for categorical features from df_diy for Streamlit
unique_categorical_values_diy = {}
for feature in categorical_features_diy:
    if feature in X_diy.columns:
        unique_categorical_values_diy[feature] = X_diy[feature].dropna().unique().tolist()
joblib.dump(unique_categorical_values_diy, os.path.join(model_dir, 'unique_cat_values_diy.pkl'))

# Also save the mean of 'Kit Cost' for default value in Streamlit for DIY model
# Check if 'Kit Cost' is still in X_diy.columns before trying to get its mean
kit_cost_mean_diy = X_diy['Kit Cost'].mean() if 'Kit Cost' in X_diy.columns else 0.0 # Default to 0.0 if not present
joblib.dump(kit_cost_mean_diy, os.path.join(model_dir, 'kit_cost_mean_diy.pkl'))

print("DIY preprocessing pipeline, model, and Streamlit metadata saved successfully.")

Epoch 1/50


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


66/66 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.8702 - loss: 0.3000 - val_accuracy: 0.9924 - val_loss: 0.0312
Epoch 2/50
66/66 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9971 - loss: 0.0184 - val_accuracy: 1.0000 - val_loss: 0.0047
Epoch 3/50
66/66 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 1.0000 - loss: 0.0032 - val_accuracy: 1.0000 - val_loss: 0.0018
Epoch 4/50
66/66 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 1.0000 - loss: 0.0017 - val_accuracy: 1.0000 - val_loss: 0.0010
Epoch 5/50
66/66 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 1.0000 - loss: 0.0010 - val_accuracy: 1.0000 - val_loss: 7.5371e-04
Epoch 6/50
66/66 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 1.0000 - loss: 9.0386e-04 - val_accuracy: 1.0000 - val_loss: 4.3297e-04
Epoch 7/50
66/66 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 1.0000 - loss: 4.3837e-04 - val_accuracy: 1.0000 - val_loss: 3.1888e-04
Epoch 8/50
66/66 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 1.0000 - loss: 3.3499e-04 - val_accuracy: 1

Once we identify the main data files (likely CSVs), we'll need to load them into a pandas DataFrame and perform exploratory data analysis (EDA). This will help us determine:

1.  **What problem we want to solve:** Is it a classification task (e.g., predicting item category), a regression task (e.g., predicting item price), or something else?
2.  **Which features are relevant:** Identify numerical, categorical, and potentially text-based features.
3.  **Data preprocessing steps:** Handling missing values, encoding categorical features, scaling numerical features, etc.

After these steps, we can then define an appropriate deep learning model architecture (e.g., a neural network for tabular data, or potentially more complex models if there are images or extensive text data).

In [10]:
!pip install --ignore-installed blinker
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 33.0 MB/s eta 0:00:00


In [13]:
!pip install streamlit pyngrok

In [23]:
from pyngrok import ngrok
ngrok.set_auth_token("3IEBnNVnGIk6M695Zuv4fqcgpTo_2XMXx8MEVKojJ6DNXFmtZ")

In [40]:
from pyngrok import ngrok

public_url = ngrok.connect(8501)
print(public_url)

NgrokTunnel: "https://unfounded-obstinate-capably.ngrok-free.dev" -> "http://localhost:8501"


In [41]:
# Run the Streamlit app
# This will provide a public URL to access the app
!streamlit run streamlit_app.py &


2026-08-21 13:56:48.183 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.75.119.159:8501

2026-08-21 13:56:56.652164: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 141ms/step


  Stopping...


In [28]:
# Kill any running ngrok processes
!killall ngrok

print("All ngrok processes killed. Please re-run the ngrok.connect() cell to establish a new tunnel.")

ngrok: no process found
All ngrok processes killed. Please re-run the ngrok.connect() cell to establish a new tunnel.
